# Robustness check: does camera normalization really help? (CPU)

The single fixed split (27 + 27 people) suggested that **camera-pair z-norm** makes CSCI's promise transfer better (97.1% promised → 95.7% delivered, vs 97.2% → 88.8% for plain similarity), and that **camera S-norm** helps the OSNet baseline. With so few people, that could be luck.

This notebook repeats the check over **25 random person-disjoint splits × 2 directions = 50 tests** per method:
- **Cell 4, summary:** the mean and spread of promised, delivered and gap, the share of "broken" promises (delivered below 93%), IDF1 and IDs per person.
- **Cell 4, paired test:** in what share of the 50 tests does normalization give a **smaller promise gap** or a **higher IDF1** than plain similarity on the **same** split?

Needs only the test fingerprints. CPU runtime, about 10 to 20 minutes. Results go to `MEVID-results/CameraNorm-SameClothes`.

In [ ]:
# 1. Load test + train fingerprints (CSCI, the OSNet baseline) and the groups
import os, glob, json, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return os.path.dirname(h[0])
    raise FileNotFoundError(pats[0])

CS = find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
           'train_templates_csci_sc.npz',
           f'{SD}/**/CSCI-SameClothes/train_templates_csci_sc.npz'])
TR = find([f'{SD}/Data/MEVID-results/Tracker-SameClothes/'
           'train_templates_tracker_sc.npz',
           f'{SD}/**/Tracker-SameClothes/train_templates_tracker_sc.npz'])
OUT = os.path.join(os.path.dirname(CS), 'CameraNorm-SameClothes')
os.makedirs(OUT, exist_ok=True)


def unit(x):
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True),
                           1e-8)).astype(np.float32)


te = np.load(f'{CS}/test_templates_csci_sc.npz')
tr = np.load(f'{CS}/train_templates_csci_sc.npz')
tte = np.load(f'{TR}/test_templates_tracker_sc.npz')
ttr = np.load(f'{TR}/train_templates_tracker_sc.npz')
pid, cam = te['pid'], te['cam']
pid_tr, cam_tr = tr['pid'], tr['cam']
DATA = {'CSCI': (unit(tr['templates']), unit(te['templates'])),
        'OSNet baseline (OSNet)': (unit(ttr['zs']), unit(tte['zs']))}
proto = json.load(open(f'{CS}/PROTOCOL_frozen_sc.json'))
fold = np.array([proto['folds'][str(int(p))] for p in pid])
SETTINGS = {'person_error_limit': 0.05, 'delta': 0.10,
            'alphas': [0.05, 0.10], 'bootstrap': 2000,
            'pair_precision_target': 0.95, 'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
print('Test clips', len(pid), '| train clips', len(pid_tr),
      '| test cameras', len(np.unique(cam)), '| results ->', OUT)


In [ ]:
# 2. Camera-aware score normalization (label-free) + ID metrics
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment


def cam_pair_znorm(S, c):
    """each score relative to the mean and spread of ALL scores between
    the same two cameras (diagonal excluded)."""
    cams = np.unique(c); ci = np.searchsorted(cams, c)
    off = ~np.eye(len(S), dtype=bool)
    Z = np.zeros_like(S)
    gmu, gsd = S[off].mean(), S[off].std()
    for a in range(len(cams)):
        for b in range(a, len(cams)):
            m = np.outer(ci == a, ci == b) | np.outer(ci == b, ci == a)
            v = S[m & off]
            mu, sd = (v.mean(), v.std()) if len(v) > 10 else (gmu, gsd)
            Z[m] = (S[m] - mu) / max(sd, 1e-6)
    return Z


def cam_snorm(S, c):
    """S-norm by camera: score of (i, j) relative to how i scores against
    j's camera and how j scores against i's camera, averaged."""
    cams = np.unique(c); ci = np.searchsorted(cams, c)
    S2 = S.astype(np.float64).copy(); np.fill_diagonal(S2, np.nan)
    mu = np.zeros((len(S), len(cams))); sd = np.ones_like(mu)
    gmu, gsd = np.nanmean(S2), np.nanstd(S2)
    for k in range(len(cams)):
        blk = S2[:, ci == k]
        cnt = np.sum(~np.isnan(blk), 1)
        m_ = np.nanmean(np.where(cnt[:, None] > 0, blk, gmu), 1)
        s_ = np.nanstd(np.where(cnt[:, None] > 0, blk, gmu), 1)
        small = cnt < 5
        m_[small], s_[small] = gmu, gsd
        mu[:, k], sd[:, k] = m_, np.maximum(s_, 1e-6)
    Zi = (S - mu[:, ci]) / sd[:, ci]
    return 0.5 * (Zi + Zi.T)


def to_dist(Z):
    """fixed (group-independent) map from normalized score to distance."""
    D = 1 / (1 + np.exp(np.clip(Z, -50, 50)))
    np.fill_diagonal(D, 0)
    return D


SCORES = {'similarity': lambda S, c: 1 - S,
          'camera-pair z-norm': lambda S, c: to_dist(cam_pair_znorm(S, c)),
          'camera S-norm': lambda S, c: to_dist(cam_snorm(S, c))}
GRIDS = {'similarity': np.round(np.arange(0.01, 0.951, 0.01), 2),
         'camera-pair z-norm': np.geomspace(1e-6, 0.5, 120),
         'camera S-norm': np.geomspace(1e-6, 0.5, 120)}


def _cont(pred, p):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1))
    np.add.at(M, (pi, ci), 1)
    return M, pi, ci


def person_losses(pred, p):
    M, _, _ = _cont(pred, p)
    nc = M.sum(0)
    tot = (M * (nc - 1)).sum(1)
    wrong = (M * (nc - M)).sum(1)
    return np.where(tot > 0, wrong / np.maximum(tot, 1), 0.0)


def score(pred, p, c):
    M, pi, ci = _cont(pred, p)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0)
    tp = (M * (M - 1)).sum()
    same_id = (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    K = np.zeros((M.shape[0], ki.max() + 1, M.shape[1]))
    np.add.at(K, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (K.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (K ** 2).sum((1, 2))).sum()
    lp = person_losses(pred, p)
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'people per ID': (M > 0).sum(0).mean(),
            'pair precision %': (100 * tp / same_id if same_id
                                 else np.nan),
            'cross-camera recall %': 100 * x_same / max(1, x_tot),
            'mean wrong-link share %': 100 * lp.mean()}


def sweep(D, p, c, grid, with_scores=True):
    D = np.clip((D + D.T) / 2, 0, None)
    Z = linkage(D[np.triu_indices(len(D), 1)], 'average')
    losses, scores = {}, {}
    for d in grid:
        lab = fcluster(Z, d, 'distance')
        losses[d] = person_losses(lab, p)
        if with_scores:
            scores[d] = score(lab, p, c)
    return losses, scores


In [ ]:
# 3. ROBUSTNESS: repeat the calibrate -> test check over many random
#    person-disjoint splits (27 people calibrate, 27 test), instead of the
#    one fixed split. For each split and direction: point threshold set on
#    the calibration people (pair precision >= 95%), applied to the test
#    people. Records promised vs delivered precision, IDF1, IDs per person.
N_SPLITS = 25
people = np.unique(pid)
rng = np.random.default_rng(SETTINGS['seed'])
rows = []
t0 = time.time()
for bb, (_, Xte) in DATA.items():
    S_all = Xte @ Xte.T
    for k in range(N_SPLITS):
        a = set(rng.choice(people, len(people) // 2, replace=False))
        grp = np.array(['A' if q in a else 'B' for q in pid])
        cache = {}
        for g in 'AB':
            ig = np.flatnonzero(grp == g)
            S = S_all[np.ix_(ig, ig)]
            cache[g] = {s: sweep(f(S, cam[ig]), pid[ig], cam[ig],
                                 GRIDS[s])[1] for s, f in SCORES.items()}
        for cal, ev in [('A', 'B'), ('B', 'A')]:
            for s in SCORES:
                cS, eS = cache[cal][s], cache[ev][s]
                d = None
                for x in GRIDS[s]:
                    pp = cS[x]['pair precision %']
                    if pp == pp and pp >= 95:
                        d = x
                if d is None:
                    continue
                rows.append({'backbone': bb, 'split': k,
                             'direction': f'{cal}->{ev}', 'score': s,
                             'promised %': cS[d]['pair precision %'],
                             'delivered %': eS[d]['pair precision %'],
                             'IDF1 %': eS[d]['IDF1 %'],
                             'IDs per person': eS[d]['IDs per person'],
                             'cross-camera recall %':
                                 eS[d]['cross-camera recall %']})
    print(f'{bb}: {N_SPLITS} splits done ({time.time() - t0:.0f} s)')
REP = pd.DataFrame(rows)
REP['gap (promised - delivered)'] = REP['promised %'] - REP['delivered %']
REP['broken (delivered < 93%)'] = (REP['delivered %'] < 93).astype(float)
REP.to_csv(f'{OUT}/robustness_random_splits.csv', index=False)


In [ ]:
# 4. Summary over all splits and directions (mean, and spread)
cols = ['promised %', 'delivered %', 'gap (promised - delivered)',
        'broken (delivered < 93%)', 'IDF1 %', 'IDs per person',
        'cross-camera recall %']
summ = REP.groupby(['backbone', 'score'], sort=False)[cols].agg(
    ['mean', 'std']).round(2)
summ.to_csv(f'{OUT}/robustness_summary.csv')
print(f'POINT THRESHOLD, {REP.split.nunique()} random splits x 2 '
      'directions (held-out people)')
display(summ)

# paired comparison against plain similarity, split by split
print('\nPaired with plain similarity (same split and direction):')
key = ['backbone', 'split', 'direction']
base = REP[REP.score == 'similarity'].set_index(key)
out = []
for s in [x for x in SCORES if x != 'similarity']:
    o = REP[REP.score == s].set_index(key)
    j = base.join(o, lsuffix='_sim', rsuffix='_norm', how='inner')
    for bb, jj in j.groupby(level=0):
        dg = (jj['gap (promised - delivered)_sim'].abs()
              - jj['gap (promised - delivered)_norm'].abs())
        di = jj['IDF1 %_norm'] - jj['IDF1 %_sim']
        out.append({'backbone': bb, 'score': s, 'pairs': len(jj),
                    'smaller gap: share of splits %':
                        round(100 * (dg > 0).mean(), 1),
                    'mean gap reduction (points)': round(dg.mean(), 2),
                    'higher IDF1: share of splits %':
                        round(100 * (di > 0).mean(), 1),
                    'mean IDF1 change (points)': round(di.mean(), 2)})
paired = pd.DataFrame(out)
paired.to_csv(f'{OUT}/robustness_paired.csv', index=False)
display(paired)
